In [5]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/distillation/medmcqa_distilled_cot_deepseek_v3.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/distillation/medmcqa_distilled_cot_deepseek_v3.parquet
Shape: (10000, 10)


,question,options,answer,answer_index,question_id,subject_name,choice_type,distill_ans_correct,distill_reasoning,distill_answer
0,Which of the following has impoant role in ini...,"['FGF-8', 'BMP-4', 'EGRF', 'Nodal gene']",D,3,9e19aa8f-d821-47e8-9766-90c626f2e6ba,Anatomy,single,False,The primitive streak is a key structure in ear...,a
1,First trimester USG finding in Down syndrome?,"['Nuchal thickening', 'Nuchal translucency', '...",B,1,f6103fcd-0018-4a28-be6f-2c6f13b62b29,Pediatrics,single,True,Down syndrome screening in the first trimester...,b


## Accuracy


In [6]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 0.8552 (8552/10000)


## Missing distilled reasoning traces


In [7]:
missing_nan = int(df["distill_answer"].isna().sum())
empty_mask = df["distill_answer"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 8


## Reasoning trace length distribution (chars)


In [8]:
existing_reasoning = df.loc[~empty_mask, "distill_reasoning"]
lengths = existing_reasoning.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count     9992.000000
mean      1049.409828
std        971.192567
min         95.000000
25%        644.000000
50%        859.000000
75%       1132.000000
max      18346.000000
Name: distill_reasoning, dtype: float64
P60: 958.0
P70: 1065.0
P80: 1208.0
P85: 1319.0
P90: 1494.8999999999996
P92: 1603.0
P95: 1978.3499999999967
P99: 6058.09


In [166]:
corrected_empty_mask = df["corrected_reasoning"].fillna("").str.strip() == ""
existing_corrected = df.loc[~corrected_empty_mask, "corrected_reasoning"]
lengths = existing_corrected.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count      25.000000
mean      735.560000
std       218.947878
min       371.000000
25%       554.000000
50%       725.000000
75%       900.000000
max      1199.000000
Name: corrected_reasoning, dtype: float64
P60: 786.1999999999999
P70: 825.5999999999999
P80: 932.8000000000002
P85: 991.6
P90: 1001.8
P92: 1008.6000000000001
P95: 1040.9999999999998
P99: 1163.2399999999998


## Reasoning trace length distribution (tokens)

Token counts per reasoning trace under each of the three model tokenizers we use: `llama3b` (`meta-llama/Llama-3.2-3B-Instruct`), `phi4mini` (`microsoft/Phi-4-mini-instruct`), and `qwen3b` (`Qwen/Qwen2.5-3B-Instruct`).

In [ ]:
# from tqdm.auto import tqdm
# from transformers import AutoTokenizer

# MODEL_TOKENIZERS = {
#     "llama3b": "meta-llama/Llama-3.2-3B-Instruct",
#     "phi4mini": "microsoft/Phi-4-mini-instruct",
#     "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
# }

# reasoning_texts = existing_reasoning.tolist()

# for name, model_id in MODEL_TOKENIZERS.items():
#     tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
#     token_lengths = pd.Series(
#         [len(tokenizer(text, add_special_tokens=False)["input_ids"]) for text in tqdm(reasoning_texts, desc=name)],
#         index=existing_reasoning.index,
#     )
#     print(f"=== {name} ({model_id}) ===")
#     print(token_lengths.describe())
#     print(f"P60: {token_lengths.quantile(0.6)}")
#     print(f"P70: {token_lengths.quantile(0.7)}")
#     print(f"P80: {token_lengths.quantile(0.8)}")
#     print(f"P85: {token_lengths.quantile(0.85)}")
#     print(f"P90: {token_lengths.quantile(0.9)}")
#     print(f"P92: {token_lengths.quantile(0.92)}")
#     print(f"P95: {token_lengths.quantile(0.95)}")
#     print(f"P99: {token_lengths.quantile(0.99)}")
#     print()

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [10]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


# n_opts_series = df["options"].apply(n_options)
valid_mask = df["distill_ans_correct"]
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 1448 (14.48%)


### All invalid answers


In [11]:
invalid_answers = df.loc[~valid_mask, "distill_reasoning"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

--- row 0 ---
'The primitive streak is a key structure in early embryonic development that establishes the body axis and leads to gastrulation. Its initiation and maintenance involve several signaling molecules.\n\n- FGF-8 (Fibroblast Growth Factor-8) is known to play a crucial role in the initiation and maintenance of the primitive streak. It is expressed in the primitive streak and helps regulate cell migration and differentiation during gastrulation.\n- BMP-4 (Bone Morphogenetic Protein-4) is involved in mesoderm formation but is more associated with ventral patterning rather than the initiation of the primitive streak itself.\n- EGRF (Epidermal Growth Factor Receptor) is involved in cell proliferation and differentiation but is not primarily linked to primitive streak function.\n- Nodal gene is important for establishing the primitive streak and left-right asymmetry, but its role is more in the initial induction rather than maintenance; FGF-8 is critical for sustaining the primitiv

### 10 shortest reasoning traces


In [12]:
shortest = existing_reasoning.str.len().nsmallest(30)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_reasoning"])
    print()

--- row 8052 (len=95) ---
Myotonic dystrophy is caused by a mutation in the DMPK gene, which is located on chromosome 19.

--- row 67 (len=116) ---
BRCA1 is located on chromosome 17, and BRCA2 is located on chromosome 13. Therefore, the correct order is 17 and 13.

--- row 2998 (len=141) ---
The diameter of the female urethra is typically about 6 mm. This is a standard anatomical measurement and is commonly cited in medical texts.

--- row 910 (len=147) ---
The macula lutea is a region in the retina responsible for high-acuity vision. Its diameter is approximately 1.5 mm, which corresponds to option a.

--- row 5130 (len=157) ---
Pellagra is a disease characterized by symptoms such as dermatitis, diarrhea, and dementia. It is caused by a deficiency of niacin, also known as vitamin B3.

--- row 3227 (len=158) ---
The pharynx extends from the base of the skull to the lower border of the cricoid cartilage anteriorly and the inferior border of the C6 vertebra posteriorly.

--- row 4307 (le

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [140]:
invalid_idx = df.index[~valid_mask]
df.loc[invalid_idx, "distill_reasoning"] = ""
df.loc[invalid_idx, "distill_answer"] = ""
df.loc[invalid_idx, "distill_ans_correct"] = False

df.to_parquet(DATASET_PATH, index=False)
print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

Reset 38 rows and wrote ../../data/out/splits/random/gpqa/train_distilled_deepseek_v4_flash_extend_w_v4_pro_short.parquet


## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [141]:
short_mask = df["distill_reasoning"].fillna("").str.len() < 100
short_idx = df.index[short_mask]
df.loc[short_idx, "distill_reasoning"] = ""
df.loc[short_idx, "distill_answer"] = ""
df.loc[short_idx, "distill_ans_correct"] = False
print(f"distill_reasoning rows under {200} chars: {len(short_idx)}")

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")

distill_reasoning rows under 200 chars: 38


In [13]:
max_reasoning_chars = 10000
long_mask = df["distill_reasoning"].str.len() > max_reasoning_chars
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
# df.loc[long_mask, "distill_reasoning"] = (
#     df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
# )
df.loc[long_mask, "distill_reasoning"] = ""
df.loc[long_mask, "distill_answer"] = ""
df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 10000 chars: 20


In [14]:
missing_reasoning_mask = df["distill_reasoning"].fillna("").eq("")
print(f"Dropping {int(missing_reasoning_mask.sum())} rows with NA or empty distill_reasoning")
df = df.loc[~missing_reasoning_mask].copy()

Dropping 20 rows with NA or empty distill_reasoning


In [158]:
max_reasoning_chars = 6000
long_mask = (df["distill_reasoning"].str.len() > max_reasoning_chars) & ~valid_mask
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
df.loc[long_mask, "distill_reasoning"] = (
    df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
)
# df.loc[long_mask, "distill_reasoning"] = ""
# df.loc[long_mask, "distill_answer"] = ""
# df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 6000 chars: 0


In [167]:
# max_corrected_reasoning_chars = 1444
# max_combined_reasoning_chars = 8192

# df["corrected_reasoning"] = df["corrected_reasoning"].str.slice(
#     stop=max_corrected_reasoning_chars
# )
has_corrected_reasoning = df["corrected_reasoning"].fillna("").str.strip().ne("")
df.loc[has_corrected_reasoning, "distill_reasoning"] = [
    reasoning + corrected
    for reasoning, corrected in zip(
        df.loc[has_corrected_reasoning, "distill_reasoning"],
        df.loc[has_corrected_reasoning, "corrected_reasoning"],
    )
]

In [8]:
df["distill_reasoning"] = df["distill_answer"]

In [9]:
from core.utils.cot_answers import extract_math_answer

df["distill_answer"] = [extract_math_answer(assistant_response, []) for assistant_response in df["distill_answer"]]


In [15]:
df.head()


,question,options,answer,answer_index,question_id,subject_name,choice_type,distill_ans_correct,distill_reasoning,distill_answer
0,Which of the following has impoant role in ini...,"['FGF-8', 'BMP-4', 'EGRF', 'Nodal gene']",D,3,9e19aa8f-d821-47e8-9766-90c626f2e6ba,Anatomy,single,False,The primitive streak is a key structure in ear...,a
1,First trimester USG finding in Down syndrome?,"['Nuchal thickening', 'Nuchal translucency', '...",B,1,f6103fcd-0018-4a28-be6f-2c6f13b62b29,Pediatrics,single,True,Down syndrome screening in the first trimester...,b
2,Depolarization is due to,"['Influx of sodium', 'Influx of potassium', 'E...",A,0,c3269966-c90c-4c97-abb5-aacac6480af5,Anatomy,multi,True,Depolarization occurs when the membrane potent...,a
3,A 55 years old smoking presents with history o...,"['Urine microscopy and cytology', 'X-ray KUB',...",A,0,f0013e89-e17a-4684-a19b-c6fab23d5b3d,Surgery,single,False,The patient is a 55-year-old smoker presenting...,c
4,Barret's esophagus is commonly associated with...,"['Adenocarcinoma', 'Squamous cell carcinoma', ...",A,0,894d49a9-6543-4299-92cc-ad701f641b29,Medicine,single,True,Barret's esophagus is a condition where the no...,a


In [16]:
df.describe()

,answer_index
count,9980.000000
mean,1.379359
std,1.114263
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,3.000000


In [17]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 0.8562 (8545/9980)


In [18]:
df.to_parquet(
    "../../data/out/distillation/medmcqa_distilled_cot_deepseek_v3_under10k.parquet", index=False
)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")